In [25]:
import numpy as np
import pandas as pd
from google.colab import drive

In [26]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [27]:
fund_path  = '/content/drive/Shareddrives/CIS_5500_Group_Project/Datasets/Capital_IQ_Data/Fundamentals_data.csv'
price_path = '/content/drive/Shareddrives/CIS_5500_Group_Project/Datasets/S&p1500_Daily.csv'

fund  = pd.read_csv(fund_path,  low_memory=False)
price = pd.read_csv(price_path, low_memory=False)

print('Fundamentals:', fund.shape)
print('Prices:',       price.shape)

Fundamentals: (39164, 27)
Prices: (7751869, 10)


## Clean Missing Values

In [28]:
# Fundamentals: drop rows missing the core join keys
print('Fundamentals before:', len(fund))
fund = fund.dropna(subset=['tic', 'gvkey', 'fyear'])
print('Fundamentals after dropping missing keys:', len(fund))

Fundamentals before: 39164
Fundamentals after dropping missing keys: 39164


In [29]:
# Drop rows missing both revt and sale, as they have no revenue data
print('Fundamentals before revenue filter:', len(fund))
fund = fund.dropna(subset=['revt', 'sale'], how='all')
print('Fundamentals after dropping rows missing both revt and sale:', len(fund))

Fundamentals before revenue filter: 39164
Fundamentals after dropping rows missing both revt and sale: 38933


In [30]:
# Prices: drop rows missing close price or date
print('Prices before:', len(price))
price = price.dropna(subset=['tic', 'gvkey', 'datadate', 'prccd'])
print('Prices after dropping missing keys/close:', len(price))

Prices before: 7751869
Prices after dropping missing keys/close: 7751803


In [31]:
# Check that there are no missing values in key columns
print('Null check - fund keys:', fund[['tic','gvkey','fyear']].isnull().sum().sum())
print('Null check - price keys:', price[['tic','gvkey','datadate','prccd']].isnull().sum().sum())

Null check - fund keys: 0
Null check - price keys: 0


## Entity Resolution

In [32]:
# Use gvkey for entity resolution
fund_gvkeys  = set(fund['gvkey'].astype(str).unique())
price_gvkeys = set(price['gvkey'].astype(str).unique())

print('Fundamentals gvkeys:', len(fund_gvkeys))
print('Price gvkeys:',        len(price_gvkeys))
print('In both:',             len(fund_gvkeys & price_gvkeys))

Fundamentals gvkeys: 1500
Price gvkeys: 1500
In both: 1500
Only in fundamentals: 0
Only in prices: 0


In [33]:
# Keep only gvkeys present in both datasets
shared_gvkeys = fund_gvkeys & price_gvkeys

fund  = fund[fund['gvkey'].astype(str).isin(shared_gvkeys)]
price = price[price['gvkey'].astype(str).isin(shared_gvkeys)]

print('Fundamentals after entity resolution:', fund.shape)
print('Prices after entity resolution:',       price.shape)

Fundamentals after entity resolution: (38933, 27)
Prices after entity resolution: (7751803, 10)


## Derive Calculated Columns

In [34]:
# Adjusted close = prccd / ajexdi
# split-adjusted price
price['adjusted_close'] = price['prccd'] / price['ajexdi'].replace(0, np.nan)
print('adjusted_close head:')
print(price[['tic', 'datadate', 'prccd', 'ajexdi', 'adjusted_close']].head())

adjusted_close sample:
  tic    datadate    prccd  ajexdi  adjusted_close
0   A  2000-01-03  72.0000     1.0         72.0000
1   A  2000-01-04  65.5625     1.0         65.5625
2   A  2000-01-05  62.3750     1.0         62.3750
3   A  2000-01-06  59.0000     1.0         59.0000
4   A  2000-01-07  65.0000     1.0         65.0000


In [35]:
# Market cap category from mkvalt ($ millions)
# Small: $100M - $1B, Mid: $1B - $10B, Large: >$10B
fund['market_cap_category'] = pd.cut(
    fund['mkvalt'],
    bins=[100, 1000, 10000, float('inf')],
    labels=['Small', 'Mid', 'Large']
)
print(fund['market_cap_category'].value_counts(dropna=False))

market_cap_category
Mid      17952
Large     8913
Small     6557
NaN       5511
Name: count, dtype: int64


## Build Lookup Tables (Sector and Industry)

In [36]:
# GIC Sector codes and names
# Source: S&P Global GIC classification standard
sector_map = {
    10: 'Energy',
    15: 'Materials',
    20: 'Industrials',
    25: 'Consumer Discretionary',
    30: 'Consumer Staples',
    35: 'Health Care',
    40: 'Financials',
    45: 'Information Technology',
    50: 'Communication Services',
    55: 'Utilities',
    60: 'Real Estate'
}

sectors_df = pd.DataFrame([
    {'gsector': k, 'sector_name': v} for k, v in sector_map.items()
])
print(sectors_df)

    gsector             sector_name
0        10                  Energy
1        15               Materials
2        20             Industrials
3        25  Consumer Discretionary
4        30        Consumer Staples
5        35             Health Care
6        40              Financials
7        45  Information Technology
8        50  Communication Services
9        55               Utilities
10       60             Real Estate


In [37]:
# GIC Industry lookup
# gind is a 6-digit code where first 2 digits match gsector
# Map the codes to the industry name
industry_map = {
    101010: 'Energy Equipment & Services',
    101020: 'Oil, Gas & Consumable Fuels',
    151010: 'Chemicals',
    151020: 'Construction Materials',
    151030: 'Containers & Packaging',
    151040: 'Metals & Mining',
    151050: 'Paper & Forest Products',
    201010: 'Aerospace & Defense',
    201020: 'Building Products',
    201030: 'Construction & Engineering',
    201040: 'Electrical Equipment',
    201050: 'Industrial Conglomerates',
    201060: 'Machinery',
    201070: 'Trading Companies & Distributors',
    202010: 'Commercial Services & Supplies',
    202020: 'Professional Services',
    203010: 'Air Freight & Logistics',
    203020: 'Passenger Airlines',
    203030: 'Marine Transportation',
    203040: 'Ground Transportation',
    203050: 'Transportation Infrastructure',
    251010: 'Automobiles & Components',
    251020: 'Automobiles',
    252010: 'Household Durables',
    252020: 'Leisure Products',
    252030: 'Textiles, Apparel & Luxury Goods',
    253010: 'Hotels, Restaurants & Leisure',
    253020: 'Diversified Consumer Services',
    254010: 'Retailing',
    255010: 'Distributors',
    255020: 'Internet & Direct Marketing Retail',
    255030: 'Multiline Retail',
    255040: 'Specialty Retail',
    301010: 'Food & Staples Retailing',
    302010: 'Beverages',
    302020: 'Food Products',
    302030: 'Tobacco',
    303010: 'Household Products',
    303020: 'Personal Products',
    351010: 'Health Care Equipment & Supplies',
    351020: 'Health Care Providers & Services',
    351030: 'Health Care Technology',
    352010: 'Biotechnology',
    352020: 'Pharmaceuticals',
    352030: 'Life Sciences Tools & Services',
    401010: 'Banks',
    401020: 'Thrifts & Mortgage Finance',
    402010: 'Diversified Financial Services',
    402020: 'Consumer Finance',
    402030: 'Capital Markets',
    402040: 'Mortgage Real Estate Investment Trusts',
    403010: 'Insurance',
    451010: 'IT Services',
    451020: 'Software',
    451030: 'Communications Equipment',
    451040: 'Technology Hardware Storage & Peripherals',
    451050: 'Electronic Equipment Instruments & Components',
    451060: 'Semiconductors & Semiconductor Equipment',
    501010: 'Diversified Telecommunication Services',
    501020: 'Wireless Telecommunication Services',
    502010: 'Media',
    502020: 'Entertainment',
    502030: 'Interactive Media & Services',
    551010: 'Electric Utilities',
    551020: 'Gas Utilities',
    551030: 'Multi-Utilities',
    551040: 'Water Utilities',
    551050: 'Independent Power & Renewable Electricity Producers',
    601010: 'Equity Real Estate Investment Trusts',
    601020: 'Real Estate Management & Development',
    452010: 'Internet Software & Services',
    452020: 'IT Consulting & Other Services',
    452030: 'Data Processing & Outsourced Services',
    453010: 'Electronic Manufacturing Services',
    601025: 'Industrial REITs',
    601030: 'Hotel & Resort REITs',
    601040: 'Office REITs',
    601050: 'Health Care REITs',
    601060: 'Residential REITs',
    601070: 'Retail REITs',
    601080: 'Specialized REITs',
    602010: 'Real Estate Services'
}

industries_df = pd.DataFrame([
    {'gind': k, 'gsector': k // 10000, 'industry_name': v}
    for k, v in industry_map.items()
])

# Keep only industries that actually appear in our data
actual_ginds = fund['gind'].dropna().astype(int).unique()
industries_df = industries_df[industries_df['gind'].isin(actual_ginds)]
print(f'Industry codes in our data: {len(actual_ginds)}')
print(f'Industry codes mapped: {len(industries_df)}')
print(industries_df.head(10))

Industry codes in our data: 73
Industry codes mapped: 73
     gind  gsector                industry_name
0  101010       10  Energy Equipment & Services
1  101020       10  Oil, Gas & Consumable Fuels
2  151010       15                    Chemicals
3  151020       15       Construction Materials
4  151030       15       Containers & Packaging
5  151040       15              Metals & Mining
6  151050       15      Paper & Forest Products
7  201010       20          Aerospace & Defense
8  201020       20            Building Products
9  201030       20   Construction & Engineering


In [38]:
mapped_ginds = set(industries_df['gind'].tolist())
unmapped = [g for g in actual_ginds if g not in mapped_ginds]
print(sorted(unmapped))

[]


## Find Index for Each Table

In [39]:
# companies: ticker should be unique
print('ticker unique in companies:', len(companies['ticker'].unique()) == len(companies))

# financial_statements: (ticker, fyear) should be unique
grouped_fund = fund.groupby(['tic', 'fyear']).size()
print('(tic, fyear) unique in fundamentals:', (grouped_fund == 1).all())

# stock_prices: (ticker, datadate) should be unique
grouped_price = price.groupby(['tic', 'datadate']).size()
print('(tic, datadate) unique in prices:', (grouped_price == 1).all())

ticker unique in companies: True
(tic, fyear) unique in fundamentals: False
(tic, datadate) unique in prices: True


In [40]:
# Look at duplicates in fundamentals
dupes = fund.groupby(['tic', 'fyear']).size()
dupes = dupes[dupes > 1].reset_index(name='count')
print(f'Duplicate (tic, fyear) pairs: {len(dupes)}')
print(dupes.head(20))

Duplicate (tic, fyear) pairs: 6726
     tic  fyear  count
0   AAMI   2021      2
1   AAMI   2022      2
2   AAMI   2023      2
3   AAMI   2024      2
4    AAT   2009      2
5    AAT   2010      2
6    AAT   2011      2
7    AAT   2012      2
8    AAT   2013      2
9    AAT   2014      2
10   AAT   2015      2
11   AAT   2016      2
12   AAT   2017      2
13   AAT   2018      2
14   AAT   2019      2
15   AAT   2020      2
16   AAT   2021      2
17   AAT   2022      2
18   AAT   2023      2
19   AAT   2024      2


In [41]:
# Same company, same year, different indfmt
sample_tic = 'AAT'
sample_year = 2020
print(fund[(fund['tic'] == sample_tic) & (fund['fyear'] == sample_year)][['tic', 'fyear', 'datadate', 'indfmt', 'consol', 'ni', 'revt']])

       tic  fyear    datadate indfmt consol      ni     revt
38268  AAT   2020  2020-12-31     FS      C     NaN  346.081
38269  AAT   2020  2020-12-31   INDL      C  28.043  345.009


In [42]:
# Keep only INLD format
# Remove duplicate rows
fund = fund[fund['indfmt'] == 'INDL']

grouped_fund = fund.groupby(['tic', 'fyear']).size()
print('(tic, fyear) unique after indfmt filter:', (grouped_fund == 1).all())
print('Fundamentals shape after filter:', fund.shape)

(tic, fyear) unique after indfmt filter: True
Fundamentals shape after filter: (32203, 28)


## Build Companies Table

In [43]:
# One row per company using most recent fiscal year record
latest_fund = fund.sort_values('fyear').groupby('gvkey').last().reset_index()

companies = latest_fund[['tic', 'gvkey', 'conm', 'gsector', 'gind', 'sic', 'mkvalt', 'market_cap_category']].copy()
companies = companies.rename(columns={'tic': 'ticker', 'conm': 'company_name', 'mkvalt': 'market_cap'})

# Confirm uniqueness
grouped = companies.groupby(['ticker', 'gvkey']).size()
print('Unique (ticker, gvkey) pairs:', (grouped == 1).all())
print('Companies shape:', companies.shape)
companies.head()

Unique (ticker, gvkey) pairs: True
Companies shape: (1500, 8)


,ticker,gvkey,company_name,gsector,gind,sic,market_cap,market_cap_category
0,AIR,1004,AAR CORP,20,201010,5080,2200.3203,Mid
1,AAL,1045,AMERICAN AIRLINES GROUP INC,20,203020,4512,10122.4143,Large
2,PNW,1075,PINNACLE WEST CAPITAL CORP,55,551010,4911,10724.1848,Large
3,PRG,1076,PROG HOLDINGS INC,40,402020,6141,1167.0962,Mid
4,ABT,1078,ABBOTT LABORATORIES,35,351010,3845,217578.4887,Large


## Export Clean CSVs

In [44]:
output_path = '/content/drive/Shareddrives/CIS_5500_Group_Project/Datasets/Cleaned/'

import os
os.makedirs(output_path, exist_ok=True)

# companies.csv
companies.to_csv(output_path + 'companies.csv', index=False)

# financial_statements.csv
fin_cols = ['tic', 'gvkey', 'datadate', 'fyear', 'revt', 'sale', 'ni',
            'oiadp', 'gp', 'ebitda', 'at', 'lt', 'ceq', 'oancf',
            'xrd', 'xsga', 'mkvalt', 'prcc_f']
financial_statements = fund[fin_cols].copy()
financial_statements.to_csv(output_path + 'financial_statements.csv', index=False)

# stock_prices.csv
price_cols = ['tic', 'gvkey', 'datadate', 'prcod', 'prchd', 'prcld',
              'prccd', 'cshtrd', 'adjusted_close', 'cshoc']
stock_prices = price[price_cols].copy()
stock_prices.to_csv(output_path + 'stock_prices.csv', index=False)

# lookup tables
sectors_df.to_csv(output_path + 'sectors.csv', index=False)
industries_df.to_csv(output_path + 'industries.csv', index=False)


Exported:
  companies.csv: 0.1 MB
  financial_statements.csv: 3.9 MB
  stock_prices.csv: 534.5 MB
  sectors.csv: 0.0 MB
  industries.csv: 0.0 MB


In [46]:
gvkey_ticker_count = companies.groupby('gvkey')['ticker'].count()
multi = gvkey_ticker_count[gvkey_ticker_count > 1]
print(f'gvkeys with multiple tickers: {len(multi)}')
print(companies[companies['gvkey'].isin(multi.index)].sort_values('gvkey')[['ticker', 'gvkey', 'company_name']].head(20))

gvkeys with multiple tickers: 0
Empty DataFrame
Columns: [ticker, gvkey, company_name]
Index: []
